In [1]:
import pandas as pd 

df = pd.read_csv("ARDS_csns.txt")
basepath = "/hpc/group/kamaleswaranlab/Emory_Deid_Tables_Tilendra/"
d1 = pd.read_csv(basepath+"SepsisInduced_ARF_Phenotypes/ARF_features_before_imputation_MICUemory_v4_matching_new_Latest.dsv", sep='|') 

In [2]:
import hashlib
from pathlib import Path 
import os 
import numpy as np

def hash_value(value, hash_key):
    return hashlib.sha256((str(value) + hash_key).encode()).hexdigest()



In [22]:
import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoModel, AutoTokenizer


# ──────────────────────────────────────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────────────────────────────────────

MODEL_NAME = "microsoft/BiomedVLP-CXR-BERT-specialized"
BASE_SAVE_DIR = Path("/work/ma618/arf_note_embeddings")

SAVE_PATHS = {
    "24_hr_most_recent": BASE_SAVE_DIR / "24_hr_most_recent",
    "48_hr_most_recent": BASE_SAVE_DIR / "48_hr_most_recent",
    "24_hr_aggregated": BASE_SAVE_DIR / "24_hr_aggregated",
    "48_hr_aggregated": BASE_SAVE_DIR / "48_hr_aggregated",
    "24_hr_concatenated": BASE_SAVE_DIR / "24_hr_concatenated",
    "48_hr_concatenated": BASE_SAVE_DIR / "48_hr_concatenated",
}


In [23]:


# ──────────────────────────────────────────────────────────────────────────────
# Model loading
# ──────────────────────────────────────────────────────────────────────────────

def load_model():
    CACHE_DIR = "/hpc/dctrl/ma618/hf_cache"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, cache_dir=CACHE_DIR)
    model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, device_map="auto", cache_dir=CACHE_DIR)
    model.eval()
    return tokenizer, model


# ──────────────────────────────────────────────────────────────────────────────
# Embedding helpers
# ──────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def embed_text(text: str, tokenizer, model) -> np.ndarray:
    """Embed a single text string using CXR-BERT. Returns a 1-D numpy array."""
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model(**inputs)
    cls_embedding = output.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
    return cls_embedding


@torch.no_grad()
def embed_texts_batch(texts: list[str], tokenizer, model, batch_size: int = 32) -> np.ndarray:
    """Embed a list of texts. Returns array of shape (N, hidden_dim)."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        output = model(**inputs)
        cls_embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
    return np.vstack(all_embeddings)


# ──────────────────────────────────────────────────────────────────────────────
# Window filtering
# ──────────────────────────────────────────────────────────────────────────────

def get_notes_in_window(df_csn: pd.DataFrame, vent_start: pd.Timestamp, hours: int) -> pd.DataFrame:
    """Return notes within [-hours, 0] relative to vent_start, sorted by timestamp ascending."""
    window_start = vent_start - pd.Timedelta(hours=hours)
    mask = (df_csn["timestamp"] >= window_start) & (df_csn["timestamp"] <= vent_start)
    return df_csn.loc[mask].sort_values("timestamp").reset_index(drop=True)


# ──────────────────────────────────────────────────────────────────────────────
# Embedding strategies
# ──────────────────────────────────────────────────────────────────────────────

def compute_most_recent_embedding(notes_window: pd.DataFrame, tokenizer, model):
    """Embed the most recent note in the window."""
    if notes_window.empty:
        return None
    most_recent_text = notes_window.iloc[-1]["notes"]
    return embed_text(most_recent_text, tokenizer, model)


def compute_aggregated_embedding(
    notes_window: pd.DataFrame, vent_start: pd.Timestamp, window_hours: int, tokenizer, model
):
    """Weighted average of note embeddings; weights = inverse time-to-vent, normalized."""
    if notes_window.empty:
        return None

    time_diffs_hrs = (vent_start - notes_window["timestamp"]).dt.total_seconds() / 3600.0
    # Clamp minimum to avoid division by zero for notes exactly at vent_start
    time_diffs_hrs = time_diffs_hrs.clip(lower=1e-3)
    raw_weights = 1.0 / time_diffs_hrs
    weights = (raw_weights / raw_weights.sum()).values

    embeddings = embed_texts_batch(notes_window["notes"].tolist(), tokenizer, model)
    weighted_embedding = np.average(embeddings, axis=0, weights=weights)
    return weighted_embedding


def compute_concatenated_embedding(notes_window: pd.DataFrame, tokenizer, model):
    """Concatenate all notes chronologically, then embed the combined text."""
    if notes_window.empty:
        return None
    concatenated_text = " ".join(notes_window["notes"].tolist())
    return embed_text(concatenated_text, tokenizer, model)


# ──────────────────────────────────────────────────────────────────────────────
# Save utility
# ──────────────────────────────────────────────────────────────────────────────

def save_embedding(embedding: np.ndarray, save_dir: Path, csn: str):
    save_dir.mkdir(parents=True, exist_ok=True)
    np.save(save_dir / f"{csn}.npy", embedding)


# ──────────────────────────────────────────────────────────────────────────────
# Main pipeline
# ──────────────────────────────────────────────────────────────────────────────

def process_all_csns(df: pd.DataFrame, tokenizer, model):
    """Generate and save embeddings for every CSN across all strategies and windows."""
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["vent_start"] = pd.to_datetime(df["vent_start"])

    for csn, group in df.groupby("csn"):
        vent_start = group["vent_start"].iloc[0]

        for hours, label in [(24, "24_hr"), (48, "48_hr")]:
            window = get_notes_in_window(group, vent_start, hours)
            if window.empty:
                continue

            # Most recent
            emb = compute_most_recent_embedding(window, tokenizer, model)
            if emb is not None:
                save_embedding(emb, SAVE_PATHS[f"{label}_most_recent"], str(csn))

            # Aggregated (weighted average)
            emb = compute_aggregated_embedding(window, vent_start, hours, tokenizer, model)
            if emb is not None:
                save_embedding(emb, SAVE_PATHS[f"{label}_aggregated"], str(csn))

            # Concatenated
            emb = compute_concatenated_embedding(window, tokenizer, model)
            if emb is not None:
                save_embedding(emb, SAVE_PATHS[f"{label}_concatenated"], str(csn))

        print(f"Done: CSN {csn}")

In [4]:
tokenizer, model = load_model()

tokenizer_config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

configuration_cxrbert.py:   0%|          | 0.00/889 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/BiomedVLP-CXR-BERT-specialized:
- configuration_cxrbert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

modeling_cxrbert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/BiomedVLP-CXR-BERT-specialized:
- modeling_cxrbert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

In [5]:
emory_root =  Path("/hpc/group/kamaleswaranlab/EmoryDataset/Images/chest_xrays_processed/lookup_files")
metadata = pd.read_csv(emory_root / "metadata_Aug11.csv")

/tmp/ipykernel_1118431/3906388700.py:2: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(emory_root / "metadata_Aug11.csv")


In [6]:
emory_notes =  Path("/hpc/group/kamaleswaranlab/EmoryDataset/EMR_RAW/noPHI/")

['2016',
 'DIALYSIS_14_21.dsv',
 '2020',
 'CJSEPSIS_IN_OUT_PROCESSED_3.csv',
 'README.md',
 '2014',
 'PEACH_HD_CRRT.dsv',
 'Combined_Radiology_Notes_with_EncounterNumber_batch_deid.dsv',
 'test.out',
 'CJSEPSIS_IN_OUT_PROCESSED_7.csv',
 '2017',
 'CJSEPSIS_OUT_EO3.csv',
 '2021',
 'cxr_icu_multiple_studies.csv',
 'merged_vent_notes_cxr.pkl',
 'CJSEPSIS_IN_OUT_PROCESSED_8.csv',
 'CJSEPSIS_IN_OUT_PROCESSED_4.csv',
 'all_radiology_notes.csv',
 'CJSEPSIS_OUT_EO3.dsv',
 'JGSEPSIS_INOUTS_ALL.dsv',
 'CJSEPSIS_IN_OUT_PROCESSED_9.csv',
 '2018',
 '2019',
 '2015',
 'CJSEPSIS_ORDEREDMEDS.dsv',
 'Combined_Radiology_Notes_with_EncounterNumber_batch_deid.csv',
 'CJSEPSIS_IN_OUT_PROCESSED_2.csv',
 'CJSEPSIS_IN_OUT_PROCESSED_combined.csv',
 'CJSEPSIS_IN_OUT_PROCESSED_5.csv',
 'VentFlowSheets',
 '2022',
 'dialysis_data.csv',
 'blood_transfusions.csv',
 'CJSEPSIS_IN_OUT_PROCESSED_0.csv',
 'CJSEPSIS_IN_OUT_PROCESSED_6.csv',
 'CJSEPSIS_IN_OUT_PROCESSED_1.csv']

In [7]:
deid_notes = pd.read_csv(emory_notes / "Combined_Radiology_Notes_with_EncounterNumber_batch_deid.dsv", sep = "|")
len(deid_notes)

/tmp/ipykernel_1118431/1001424604.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  all_notes = pd.read_csv(emory_notes / "all_radiology_notes.csv")
/tmp/ipykernel_1118431/1001424604.py:2: DtypeWarning: Columns (0,2,3,4,5,6,7,8,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  deid_notes = pd.read_csv(emory_notes / "Combined_Radiology_Notes_with_EncounterNumber_batch_deid.dsv", sep = "|")


(461181, 3024642)

In [8]:
arf_csns_to_vent_start = dict(zip(d1['csn'], d1['vent_start_time']))
len(arf_csns_to_vent_start)

3349

In [9]:
def safe_hash(x, salt='123'):
    if pd.isna(x):
        return np.nan
    return hash_value(str(np.int32(x)), salt)

deid_notes['DAY_VERIFIED'] = pd.to_datetime(deid_notes['DAY_VERIFIED'])


In [10]:
deid_notes.columns

Index(['Unnamed: 0', 'ACC_NBR', 'PATIENT_ID', 'EMPI_NBR', 'ENCNTR_ID',
       'HNAM_DOCUMENT_CLINICAL_ID', 'HNAM_DOCUMENT_CLINICAL_NM',
       'DAY_VERIFIED', 'EVENT_DOCUMENT_DESC', 'EVENT_DOCUMENT_KEY',
       'ENCOUNTER_ID', 'ENCOUNTER_NBR', 'notes_deid'],
      dtype='object')

In [11]:
arf_notes = deid_notes.loc[deid_notes.ENCOUNTER_NBR.isin(arf_csns_to_vent_start.keys())]

In [12]:
arf_notes_cxr = arf_notes.loc[arf_notes.HNAM_DOCUMENT_CLINICAL_NM.str.lower().str.contains('xr chest')]
len(arf_notes_cxr)

29797

In [13]:
arf_notes_cxr['vent_start'] = pd.to_datetime(arf_notes_cxr['ENCOUNTER_NBR'].map(arf_csns_to_vent_start))


/tmp/ipykernel_1118431/3927628838.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arf_notes_cxr['vent_start'] = pd.to_datetime(arf_notes_cxr['ENCOUNTER_NBR'].map(arf_csns_to_vent_start))


In [14]:
arf_notes_cxr['time_diff'] = (arf_notes_cxr['DAY_VERIFIED'] - arf_notes_cxr['vent_start']).dt.total_seconds()/3600

/tmp/ipykernel_1118431/3995969242.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arf_notes_cxr['time_diff'] = (arf_notes_cxr['DAY_VERIFIED'] - arf_notes_cxr['vent_start']).dt.total_seconds()/3600


In [15]:
arf_notes_cxr["qualifying_notes_24"] = (arf_notes_cxr["time_diff"] >= -24) & (arf_notes_cxr["time_diff"] < 0)
arf_notes_cxr["qualifying_notes_48"] = (arf_notes_cxr["time_diff"] >= -48) & (arf_notes_cxr["time_diff"] < 0) 
arf_notes_cxr["qualifying_notes_48"].sum(), arf_notes_cxr["qualifying_notes_24"].sum() 

/tmp/ipykernel_1118431/3302946403.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arf_notes_cxr["qualifying_notes_24"] = (arf_notes_cxr["time_diff"] >= -24) & (arf_notes_cxr["time_diff"] < 0)
/tmp/ipykernel_1118431/3302946403.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arf_notes_cxr["qualifying_notes_48"] = (arf_notes_cxr["time_diff"] >= -48) & (arf_notes_cxr["time_diff"] < 0)


(np.int64(6079), np.int64(4716))

In [16]:
arf_notes_cxr.loc[arf_notes_cxr.qualifying_notes_24].groupby('ENCOUNTER_NBR')['ACC_NBR'].nunique().describe()

count    2801.000000
mean        1.683684
std         0.812348
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         6.000000
Name: ACC_NBR, dtype: float64

In [17]:
arf_notes_cxr.loc[arf_notes_cxr.qualifying_notes_48].groupby('ENCOUNTER_NBR')['ACC_NBR'].nunique().describe()

count    3041.000000
mean        1.999013
std         0.993233
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         7.000000
Name: ACC_NBR, dtype: float64

In [25]:
arf_notes_cxr.columns

Index(['Unnamed: 0', 'ACC_NBR', 'PATIENT_ID', 'EMPI_NBR', 'ENCNTR_ID',
       'HNAM_DOCUMENT_CLINICAL_ID', 'HNAM_DOCUMENT_CLINICAL_NM', 'timestamp',
       'EVENT_DOCUMENT_DESC', 'EVENT_DOCUMENT_KEY', 'ENCOUNTER_ID',
       'ENCOUNTER_NBR', 'notes', 'vent_start', 'time_diff',
       'qualifying_notes_24', 'qualifying_notes_48'],
      dtype='object')

In [26]:
arf_notes_cxr.rename(columns = {
    'ENCOUNTER_NBR': 'csn'
}, inplace = True)

/tmp/ipykernel_1118431/1803238401.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arf_notes_cxr.rename(columns = {


In [21]:
arf_notes_cxr.rename(columns = {
    'DAY_VERIFIED': 'timestamp',
    'notes_deid': 'notes'
}, inplace = True)

/tmp/ipykernel_1118431/4005023749.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  arf_notes_cxr.rename(columns = {


In [27]:
process_all_csns(arf_notes_cxr, tokenizer, model)

/tmp/ipykernel_1118431/1036959517.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["timestamp"] = pd.to_datetime(df["timestamp"])
/tmp/ipykernel_1118431/1036959517.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["vent_start"] = pd.to_datetime(df["vent_start"])


Done: CSN 006618cf56a60cee8f2ae1e7812f08a0de73526dd14defed7307e85cf3e972b9
Done: CSN 0068d0b7dc1b083df6ec7e57cece8884e61c4ecee80af02cdc8c43d60d59e7ea
Done: CSN 006e60a283acfeee09fac4a422e0a099ffdca3d42cd830f8fdaf37700b753a70
Done: CSN 00773fe2622a31d47cb56e452d6c3e720e546815ad572d7b7532edf10002b3b0
Done: CSN 007b0600e4f9a88ebe9bac292d08b989728683a86707fddf85f54282f549ae02
Done: CSN 00868cbad17e89151393beef0047179b2b72e6f4b250489fbec89d70fda69bfa
Done: CSN 00b6d1fc390d616eccf8b7a4b56ac3af0a483b17c35779d91c5a00bf2d883911
Done: CSN 00c740052a52e61a2ff822547d14f8d84d168cd26eab094bac76a58a50db19e7
Done: CSN 00ef31c089afde79224422a00e1f7deedb3305bb2fc38029f5678e8847979b3f
Done: CSN 010ba843608bfcfa102661ab1b23f32d90bf719c03268c97307f00f2976aa121
Done: CSN 0134672b2f9a0fef53d4cb98f0095646ab7eb00d887a05f30895cf93249b025a
Done: CSN 0138cd6335db299b69077aca34d13ed0a4e209500dabb34ecb67ae91a7689309
Done: CSN 01495087c9581b3e1d10c4ed8c6ca67235615e447a23377156b9734e18aa1eb1
Done: CSN 01681103faa23bc